In [261]:
# Importing necessary libraries
import pandas as pd
import numpy as np

In [262]:
# Reading the CSV file into a DataFrame
df =  pd.read_csv(r"C:\Users\v-vvajpai\Desktop\Project_2\datasets\consumer-price-index.csv")

In [263]:
# Checking the shape of the DataFrame
df.shape

(291890, 8)

In [264]:
# Checking the data types of each column in the DataFrame
df.dtypes

id                   int64
date                   str
state_name             str
state_code           int64
commodity_group        str
sector                 str
cpi                float64
inflation_rate     float64
dtype: object

In [265]:
# converting the 'date' column to datetime format for easier manipulation and analysis
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

In [266]:
df.dtypes

id                          int64
date               datetime64[us]
state_name                    str
state_code                  int64
commodity_group               str
sector                        str
cpi                       float64
inflation_rate            float64
dtype: object

In [267]:
# Inspecting the first 20 rows of the DataFrame
df.head(20)

,id,date,state_name,state_code,commodity_group,sector,cpi,inflation_rate
0,0,2014-01-01,All India,0,Cereals And Products,Combined,119.6,10.33
1,1,2014-01-01,All India,0,Cereals And Products,Rural,118.9,10.60
2,2,2014-01-01,All India,0,Cereals And Products,Urban,121.2,9.68
3,3,2014-01-01,All India,0,Clothing,Combined,115.8,8.94
4,4,2014-01-01,All India,0,Clothing,Rural,116.5,9.39
5,5,2014-01-01,All India,0,Clothing,Urban,114.8,8.40
6,6,2014-01-01,All India,0,Clothing And Footwear,Combined,115.4,8.66
7,7,2014-01-01,All India,0,Clothing And Footwear,Rural,116.2,9.21
8,8,2014-01-01,All India,0,Clothing And Footwear,Urban,114.3,8.03
9,9,2014-01-01,All India,0,Consumer Food Price Index,Combined,115.6,9.68


 For this data, one row = one (state + commodity + sector) measured in one month.

In [268]:
# Checking missing values in the DataFrame
df.isna().sum()

id                    0
date                  0
state_name            0
state_code            0
commodity_group       0
sector                0
cpi                2822
inflation_rate     2822
dtype: int64

In [269]:
# Establishing the relationship between missing values in 'cpi' and 'inflation_rate'

both_missing = df["cpi"].isna() & df["inflation_rate"].isna()
cpi_only_missing = df["cpi"].isna() & df["inflation_rate"].notna()
inflation_only_missing = df["cpi"].notna() & df["inflation_rate"].isna()

print("Both missing:", both_missing.sum())
print("Only CPI missing:", cpi_only_missing.sum())
print("Only inflation_rate missing:", inflation_only_missing.sum())
print("Total rows with any missing:", (both_missing | cpi_only_missing | inflation_only_missing).sum())

Both missing: 2822
Only CPI missing: 0
Only inflation_rate missing: 0
Total rows with any missing: 2822


In [270]:
# Inspecting the 'cpi' column for statistical summary
df['cpi'].describe()
# Min value: 0.0 remains point of concern, as it may indicate an error or a placeholder for missing data as CPI cannot be zero in a real-world. 

count    289068.000000
mean        147.934146
std          40.258153
min           0.000000
25%         125.300000
50%         145.100000
75%         173.000000
max         530.400000
Name: cpi, dtype: float64

Min Value stands at 0.. Strange as CPI cannot be 0

In [271]:
# Identifying the number of rows where 'cpi' is zero
(df.cpi==0).sum()

np.int64(8360)

In [272]:
# Inspecting the 'inflation_rate' column for statistical summary
df['inflation_rate'].describe()
# Min value: -100.0 is a point of concern, as it may indicate an error or a placeholder for missing data as inflation rate cannot be -100% in a real-world scenario.

count    289068.000000
mean          4.881925
std           6.892318
min        -100.000000
25%           1.850000
50%           4.440000
75%           7.240000
max         136.210000
Name: inflation_rate, dtype: float64

In [273]:
keys = ['state_name', 'commodity_group', 'sector']

missing = df[df['cpi'].isna()]

for key in keys:
    print(f"Missing 'cpi' values grouped by {key}:")
    print(missing.groupby(key).size().sort_values(ascending=False))

Missing 'cpi' values grouped by state_name:
state_name
Arunachal Pradesh              408
Andhra Pradesh                  68
Andaman And Nicobar Islands     68
Assam                           68
Bihar                           68
Maharashtra                     68
Chandigarh                      68
Chhattisgarh                    68
Dadra And Nagar Haveli          68
Daman And Diu                   68
Delhi                           68
Goa                             68
Gujarat                         68
Haryana                         68
Himachal Pradesh                68
Jammu And Kashmir               68
Jharkhand                       68
Karnataka                       68
Kerala                          68
Lakshadweep                     68
Madhya Pradesh                  68
Rajasthan                       68
Manipur                         68
Meghalaya                       68
Mizoram                         68
Nagaland                        68
Odisha                          68


In [274]:
df[df.cpi.isna()].groupby(df.date.dt.year).size()

date
2022    166
2023    996
2024    996
2025    664
dtype: int64

### Observations - How data is Missing?

- Missing Values start from 2022
- Missing Values **dominated** by one commodity group = **housing**
- Missing Values mostly appearing on Arunachal Pradesh state
- Missing Values mostly on Rural & Combined sector

### Why CPI = 0?

In [275]:
# Inspecting the distribution of CPI = 0 across years
df[df.cpi==0].groupby(df.date.dt.year).size()

date
2014     996
2015     996
2016     996
2017     996
2018     844
2019    1005
2020     593
2021    1021
2022     913
dtype: int64

In [276]:
# Inspecting the distribution of CPI = 0 across different commodity groups
df[df.cpi == 0].groupby('commodity_group').size()

commodity_group
Clothing                            3
Clothing And Footwear             215
Food And Beverages                196
Footwear                            4
Fuel And Light                    196
General Index (All Groups)        200
Housing                          7151
Miscellaneous                     196
Pan; Tobacco; And Intoxicants     199
dtype: int64

- CPI Value = 0 is also dominated by Housing commodity group
- CPI Value = 0 appears from 2014-22 post which it does not exist

Based on this, we can infer that housing commodity group needs to inspected further. 

#### Findings 

- Only CPI & Inflation rate column has missing values 
- Both CPI and Inflation rate are missing at same placeholders (we can inspect only cpi further)
- Commodity group 'Housing' dominated the missing value
- All the missing values of CPI are from year 2022 onwards and interestingly all the 0 value of CPI exist till 2022. 

### Investigation - Housing Commodity Group? 

In [277]:
hdf = df[df.commodity_group == 'Housing']
hdf.head(10)


,id,date,state_name,state_code,commodity_group,sector,cpi,inflation_rate
39,39,2014-01-01,All India,0,Housing,Combined,111.6,11.27
40,40,2014-01-01,All India,0,Housing,Rural,0.0,0.00
41,41,2014-01-01,All India,0,Housing,Urban,111.6,11.27
96,96,2014-01-01,Andaman And Nicobar Islands,35,Housing,Combined,0.0,0.00
97,97,2014-01-01,Andaman And Nicobar Islands,35,Housing,Rural,0.0,0.00
98,98,2014-01-01,Andaman And Nicobar Islands,35,Housing,Urban,112.3,12.30
141,141,2014-01-01,Andhra Pradesh,28,Housing,Combined,0.0,0.00
142,142,2014-01-01,Andhra Pradesh,28,Housing,Rural,0.0,0.00
143,143,2014-01-01,Andhra Pradesh,28,Housing,Urban,111.5,11.28
240,240,2014-01-01,Assam,18,Housing,Combined,0.0,0.00


In [278]:
sector_cpi_status = (
    hdf.assign(
        cpi_missing=hdf["cpi"].isna(),
        cpi_zero=hdf["cpi"].eq(0),
        cpi_missing_or_zero=hdf["cpi"].isna() | hdf["cpi"].eq(0)
    )
    .groupby("sector")
    .agg(
        total_rows=("cpi", "size"),
        missing_cpi=("cpi_missing", "sum"),
        zero_cpi=("cpi_zero", "sum"),
        missing_or_zero_cpi=("cpi_missing_or_zero", "sum")
    )
)

sector_cpi_status["missing_or_zero_cpi_pct"] = (
    sector_cpi_status["missing_or_zero_cpi"] / sector_cpi_status["total_rows"] * 100
).round(2)

sector_cpi_status.sort_values("missing_or_zero_cpi", ascending=False)

,total_rows,missing_cpi,zero_cpi,missing_or_zero_cpi,missing_or_zero_cpi_pct
sector,,,,,
Rural,4900,1224,3676,4900,100.00
Combined,4899,1190,3475,4665,95.22
Urban,4864,0,0,0,0.00


- Rural sector for housing commodity has all it's CPI value as either 0 or missing. Given this, we can conclude it was never reported and hence it is **Structurally Missing (MNAR)**. So there is not reason to impute this CPI value rather it makes sense to drop this subset of data from the dataset. 

### Deleting the rows

In [279]:
mask = (
    (df['commodity_group'] == 'Housing') &
    (df['sector'] == 'Rural')
)
print(type(mask))
print(mask.dtype)
print("Rows to delete:", mask.sum())
print("Nan cpi values to delete:", df.loc[mask, 'cpi'].isna().sum())
print("Zero cpi values to delete:", (df.loc[mask, 'cpi'] == 0).sum())



<class 'pandas.Series'>
bool
Rows to delete: 4900
Nan cpi values to delete: 1224
Zero cpi values to delete: 3676


In [280]:
df = df[~mask].copy()


In [281]:
hdf = df[df.commodity_group == 'Housing']
hdf.head(10)


,id,date,state_name,state_code,commodity_group,sector,cpi,inflation_rate
39,39,2014-01-01,All India,0,Housing,Combined,111.6,11.27
41,41,2014-01-01,All India,0,Housing,Urban,111.6,11.27
96,96,2014-01-01,Andaman And Nicobar Islands,35,Housing,Combined,0.0,0.00
98,98,2014-01-01,Andaman And Nicobar Islands,35,Housing,Urban,112.3,12.30
141,141,2014-01-01,Andhra Pradesh,28,Housing,Combined,0.0,0.00
143,143,2014-01-01,Andhra Pradesh,28,Housing,Urban,111.5,11.28
240,240,2014-01-01,Assam,18,Housing,Combined,0.0,0.00
242,242,2014-01-01,Assam,18,Housing,Urban,109.2,8.87
321,321,2014-01-01,Bihar,10,Housing,Combined,0.0,0.00
323,323,2014-01-01,Bihar,10,Housing,Urban,111.7,11.25


### Further Investigation

In [282]:
df.isna().sum()

id                    0
date                  0
state_name            0
state_code            0
commodity_group       0
sector                0
cpi                1598
inflation_rate     1598
dtype: int64

In [283]:
missing = df[df['cpi'].isna()]

for key in keys:
    print(f"Missing 'cpi' values grouped by {key}:")
    print(missing.groupby(key).size().sort_values(ascending=False))

Missing 'cpi' values grouped by state_name:
state_name
Arunachal Pradesh              408
Andaman And Nicobar Islands     34
Andhra Pradesh                  34
Assam                           34
Bihar                           34
Chandigarh                      34
Chhattisgarh                    34
Dadra And Nagar Haveli          34
Daman And Diu                   34
Delhi                           34
Goa                             34
Gujarat                         34
Haryana                         34
Himachal Pradesh                34
Jammu And Kashmir               34
Jharkhand                       34
Karnataka                       34
Kerala                          34
Lakshadweep                     34
Madhya Pradesh                  34
Maharashtra                     34
Manipur                         34
Meghalaya                       34
Mizoram                         34
Nagaland                        34
Odisha                          34
Puducherry                      34


#### Observations

After removing Housing/Rural missing and unreal data we further inspect the data again to identify the missing value distribution and observe that:-
- Missing Value dominated by Housing commodity group
- Missing Value is concentrated mostly on combined sector 
- Missing Value mainly on Arunachal Pradesh

### Investigation

In [284]:
hc = df[(df.commodity_group == 'Housing') & (df.sector == 'Combined')]
hc.head(10)

,id,date,state_name,state_code,commodity_group,sector,cpi,inflation_rate
39,39,2014-01-01,All India,0,Housing,Combined,111.6,11.27
96,96,2014-01-01,Andaman And Nicobar Islands,35,Housing,Combined,0.0,0.00
141,141,2014-01-01,Andhra Pradesh,28,Housing,Combined,0.0,0.00
240,240,2014-01-01,Assam,18,Housing,Combined,0.0,0.00
321,321,2014-01-01,Bihar,10,Housing,Combined,0.0,0.00
378,378,2014-01-01,Chandigarh,4,Housing,Combined,0.0,0.00
423,423,2014-01-01,Chhattisgarh,22,Housing,Combined,0.0,0.00
480,480,2014-01-01,Dadra And Nagar Haveli,26,Housing,Combined,0.0,0.00
501,501,2014-01-01,Daman And Diu,25,Housing,Combined,0.0,0.00
546,546,2014-01-01,Delhi,7,Housing,Combined,0.0,0.00


In [285]:
hc_state_cpi_status = (
    hc.assign(
        cpi_missing=hc["cpi"].isna(),
        cpi_zero=hc["cpi"].eq(0)
    )
    .groupby("state_name", as_index=False)
    .agg(
        total_rows=("cpi", "size"),
        missing_cpi=("cpi_missing", "sum"),
        zero_cpi=("cpi_zero", "sum")
    )
)


hc_state_cpi_status = hc_state_cpi_status.sort_values(
    ["missing_cpi", "zero_cpi"], ascending=False
).reset_index(drop=True)
hc_state_cpi_status["missing_or_zero_cpi"] = (
    hc_state_cpi_status["missing_cpi"] + hc_state_cpi_status["zero_cpi"]
)

hc_state_cpi_status["missing_or_zero_cpi_pct"] = (
    hc_state_cpi_status["missing_or_zero_cpi"] / hc_state_cpi_status["total_rows"] * 100
).round(2)

hc_state_cpi_status["sector"] = "Combined"
hc_state_cpi_status

,state_name,total_rows,missing_cpi,zero_cpi,missing_or_zero_cpi,missing_or_zero_cpi_pct,sector
0,Andaman And Nicobar Islands,136,34,99,133,97.79,Combined
1,Andhra Pradesh,136,34,99,133,97.79,Combined
2,Assam,136,34,99,133,97.79,Combined
3,Bihar,136,34,99,133,97.79,Combined
4,Chandigarh,136,34,99,133,97.79,Combined
5,Chhattisgarh,136,34,99,133,97.79,Combined
6,Dadra And Nagar Haveli,136,34,99,133,97.79,Combined
7,Daman And Diu,136,34,99,133,97.79,Combined
8,Delhi,136,34,99,133,97.79,Combined
9,Goa,136,34,99,133,97.79,Combined


- State Name = All India has only 7.19% CPI values as 0 given this we can include this data in the dataset ad impute the other missing or 0 values. 
- For Combined Sector in Housing commodity group, we see that all the other states have **97%** of data as either missing or 0 making it unusable to impute. Given this, it is better to drop these rows from the dataset. 

### Deleting Rows

In [286]:
mask = (
    (df['commodity_group'] == 'Housing') &
    (df['sector'] == 'Combined') &
    (df['state_name'] != 'All India')
)
print(type(mask))
print(mask.dtype)
print("Rows to delete:", mask.sum())
print("Nan cpi values to delete:", df.loc[mask, 'cpi'].isna().sum())
print("Zero cpi values to delete:", (df.loc[mask, 'cpi'] == 0).sum())



<class 'pandas.Series'>
bool
Rows to delete: 4760
Nan cpi values to delete: 1190
Zero cpi values to delete: 3465


In [287]:
df = df[~mask].copy()


### Further Investigation

In [288]:
df.isna().sum()

id                   0
date                 0
state_name           0
state_code           0
commodity_group      0
sector               0
cpi                408
inflation_rate     408
dtype: int64

In [289]:
missing = df[df['cpi'].isna()]

for key in keys:
    print(f"Missing 'cpi' values grouped by {key}:")
    print(missing.groupby(key).size().sort_values(ascending=False))

Missing 'cpi' values grouped by state_name:
state_name
Arunachal Pradesh    408
dtype: int64
Missing 'cpi' values grouped by commodity_group:
commodity_group
Clothing And Footwear            68
Food And Beverages               68
Fuel And Light                   68
General Index (All Groups)       68
Miscellaneous                    68
Pan; Tobacco; And Intoxicants    68
dtype: int64
Missing 'cpi' values grouped by sector:
sector
Combined    204
Urban       204
dtype: int64


In [290]:
df['cpi'].describe()

count    281822.000000
mean        151.683863
std          33.071184
min           0.000000
25%         126.700000
50%         146.300000
75%         173.800000
max         530.400000
Name: cpi, dtype: float64

In [291]:
df[(df.cpi == 0) | (df.cpi.isna())].count()

id                 1627
date               1627
state_name         1627
state_code         1627
commodity_group    1627
sector             1627
cpi                1219
inflation_rate     1219
dtype: int64

In [292]:
df[(df.cpi == 0) | (df.cpi.isna())].groupby(['state_name']).size().sort_values(ascending=False)

state_name
Arunachal Pradesh              1588
All India                        10
Manipur                           8
Nagaland                          5
Uttarakhand                       4
Meghalaya                         4
Kerala                            3
Delhi                             2
Andaman And Nicobar Islands       1
Jharkhand                         1
Chandigarh                        1
dtype: int64

### Investigating Arunachal Pradesh

In [293]:
adf = df[(df.state_name == 'Arunachal Pradesh')]

In [294]:
# New dataframe with only Arunachal Pradesh data
adf.head(10)

,id,date,state_name,state_code,commodity_group,sector,cpi,inflation_rate
186,186,2014-01-01,Arunachal Pradesh,12,Clothing And Footwear,Combined,0.0,0.00
187,187,2014-01-01,Arunachal Pradesh,12,Clothing And Footwear,Rural,122.8,14.34
188,188,2014-01-01,Arunachal Pradesh,12,Clothing And Footwear,Urban,0.0,0.00
189,189,2014-01-01,Arunachal Pradesh,12,Food And Beverages,Combined,0.0,0.00
190,190,2014-01-01,Arunachal Pradesh,12,Food And Beverages,Rural,114.7,7.20
191,191,2014-01-01,Arunachal Pradesh,12,Food And Beverages,Urban,0.0,0.00
192,192,2014-01-01,Arunachal Pradesh,12,Fuel And Light,Combined,0.0,0.00
193,193,2014-01-01,Arunachal Pradesh,12,Fuel And Light,Rural,114.6,9.35
194,194,2014-01-01,Arunachal Pradesh,12,Fuel And Light,Urban,0.0,0.00
195,195,2014-01-01,Arunachal Pradesh,12,General Index (All Groups),Combined,0.0,0.00


In [295]:
total_rows = len(adf)

cpi_missing_count = df["cpi"].isna().sum()
cpi_zero_count = (df["cpi"] == 0).sum()
total_fake_rows = cpi_missing_count + cpi_zero_count

print(f"Total rows: {total_rows}")
print(f"CPI missing: {cpi_missing_count} ({cpi_missing_count / total_rows * 100:.2f}%)")
print(f"CPI zero: {cpi_zero_count} ({cpi_zero_count / total_rows * 100:.2f}%)")
print(f"Total fake rows (missing or zero): {total_fake_rows} ({total_fake_rows / total_rows * 100:.2f}%)")

Total rows: 2451
CPI missing: 408 (16.65%)
CPI zero: 1219 (49.73%)
Total fake rows (missing or zero): 1627 (66.38%)


Based on this we can see that Arunachal Pradesh contains 66.38% missing and non real data. 

In [296]:
# Distribution of zeroes/missing values by Commodity group
commodity_table = (
    adf.groupby("commodity_group")["cpi"]
       .apply(lambda x: round(((x.isna()) | (x == 0)).mean() * 100, 2))
       .rename("% Missing/Zero")
       .to_frame()
)

print(commodity_table)


                               % Missing/Zero
commodity_group                              
Clothing And Footwear                   64.86
Food And Beverages                      64.86
Fuel And Light                          64.86
General Index (All Groups)              64.42
Miscellaneous                           64.86
Pan; Tobacco; And Intoxicants           64.86


In [297]:
# Distribution of zeroes/missing values by sector group
sector_table = (
    adf.groupby('sector')['cpi']
    .apply(lambda x: round(((x.isna()) | (x==0)).mean()*100, 2))
    .rename("% Missing/Zero")
    .to_frame()
)
print(sector_table)

          % Missing/Zero
sector                  
Combined           93.89
Rural               0.00
Urban             100.00


We observe that the missing cpi or cpi = 0 for Arunachal Pradesh is dominated by combined and urban sector with staggering 93% and 100%. So given this, it doesn't make any sense to include these in the analysis as imputing them create a entire fake series.

In [298]:
mask_ap = (
    (df['state_name'] == 'Arunachal Pradesh') &
    ((df['sector'] == 'Combined') | 
    (df['sector'] == 'Urban'))
)
print(type(mask_ap))
print(mask_ap.dtype)
print("Rows to delete:", mask_ap.sum())
print("Nan cpi values to delete:", df.loc[mask_ap, 'cpi'].isna().sum())
print("Zero cpi values to delete:", (df.loc[mask_ap, 'cpi'] == 0).sum())



<class 'pandas.Series'>
bool
Rows to delete: 1638
Nan cpi values to delete: 408
Zero cpi values to delete: 1180


In [299]:
df = df[~mask_ap].copy()


### Final Observations

In [300]:
df.isna().sum()

id                 0
date               0
state_name         0
state_code         0
commodity_group    0
sector             0
cpi                0
inflation_rate     0
dtype: int64

No missing value exist in the dataframe

In [301]:
df['cpi'].describe()

count    280592.000000
mean        152.320947
std          31.643649
min           0.000000
25%         126.900000
50%         146.500000
75%         174.000000
max         530.400000
Name: cpi, dtype: float64

Minimum Value of CPI still shows 0 indicating there are still some values in the data with unreal values. 

In [302]:
# Number of rows with CPI = 0
cpi_zero = df[df['cpi'] == 0]
print(f"Number of rows with CPI ={len(cpi_zero)}")

Number of rows with CPI =39


In [303]:
# Occurrences of CPI = 0
(df['cpi'] == 0).sum()

np.int64(39)

In [304]:
# Distribution of zeroes/missing values
keys = ['state_name', 'commodity_group', 'sector']

missing = df[df['cpi'] == 0]

for key in keys:
    print(f"Missing 'cpi' values grouped by {key}:")
    print(missing.groupby(key).size().sort_values(ascending=False))

Missing 'cpi' values grouped by state_name:
state_name
All India                      10
Manipur                         8
Nagaland                        5
Uttarakhand                     4
Meghalaya                       4
Kerala                          3
Delhi                           2
Andaman And Nicobar Islands     1
Jharkhand                       1
Chandigarh                      1
dtype: int64
Missing 'cpi' values grouped by commodity_group:
commodity_group
Clothing And Footwear            19
Housing                          10
Footwear                          4
Clothing                          3
Pan; Tobacco; And Intoxicants     3
dtype: int64
Missing 'cpi' values grouped by sector:
sector
Rural       18
Combined    13
Urban        8
dtype: int64


In [305]:
# Occurrences of CPI = 0 over time (monthly)
df[df.cpi==0].groupby(df.date.dt.to_period('M')).size()

date
2018-12     1
2019-01     1
2019-02     1
2019-03     1
2019-04     1
2019-05     1
2019-06     1
2019-07     1
2019-08     1
2019-09     1
2020-08     4
2021-05    14
2021-06     8
2021-07     2
2021-08     1
Freq: M, dtype: int64

Most gaps fall in May–Jun 2021 (COVID Delta wave) and Aug 2020 (first wave) — periods when India's price-collection was disrupted. These are temporary reporting misses in live series, not "this item doesn't exist here." That is a fill case, not a delete case.

### Interpolating the values

In [306]:
df.dtypes

id                          int64
date               datetime64[us]
state_name                    str
state_code                  int64
commodity_group               str
sector                        str
cpi                       float64
inflation_rate            float64
dtype: object

In [307]:
print("zeroes_before:",(df['cpi'] == 0).sum())
df['cpi'] = df['cpi'].replace(0, np.nan)
print("zeroes_after:", (df['cpi'] == 0).sum())
print("missing_now:", df['cpi'].isna().sum())

zeroes_before: 39
zeroes_after: 0
missing_now: 39


In [308]:
df = df.drop_duplicates(subset=keys + ['date'])
df = df.sort_values(keys + ['date']).reset_index(drop=True)
print(df.duplicated(subset=keys + ['date']).sum(), "duplicates remain")

0 duplicates remain


In [309]:
gap_mask = df['cpi'].isna().copy()
print('gaps to fill:', gap_mask.sum())

gaps to fill: 39


def interp_cpi(g):
    g = g.sort_values('date')
    g['cpi'] = g.set_index('date')['cpi'].interpolate(method='time').values
    return g

df = df.groupby(keys, group_keys = False).apply(interp_cpi)

In [311]:
gap_mask = df['cpi'].isna().copy()
print('gaps to fill:', gap_mask.sum())

df['cpi'] = (
    df.set_index('date')
    .groupby(keys)['cpi']
    .transform(lambda s: s.interpolate(method='time'))
    .to_numpy( )
)

print("cpi missing after:", df['cpi'].isna().sum())

gaps to fill: 39
cpi missing after: 0


In [312]:
print("cpi still missing:", df['cpi'].isna().sum())
print('cpi == 0 left:', (df['cpi'] == 0).sum())
print('cpi min:', round(df['cpi'].min(), 2))

cpi still missing: 0
cpi == 0 left: 0
cpi min: 76.5


In [313]:
df.loc[gap_mask, 'inflation_rate'] = np.nan

df['inflation_rate'] = (
    df.set_index('date')
    .groupby(keys)['inflation_rate']
    .transform(lambda s: s.interpolate(method='time'))
    .to_numpy()
)

print("inflation_rate missing after:", df['inflation_rate'].isna().sum())

inflation_rate missing after: 0


In [314]:
print(df.isna().sum())        # all zeros
print("cpi==0 left:", (df['cpi']==0).sum())   # 0
print(df['cpi'].describe())

id                 0
date               0
state_name         0
state_code         0
commodity_group    0
sector             0
cpi                0
inflation_rate     0
dtype: int64
cpi==0 left: 0
count    280592.000000
mean        152.342820
std          31.593107
min          76.500000
25%         126.900000
50%         146.500000
75%         174.000000
max         530.400000
Name: cpi, dtype: float64


In [316]:
df.to_csv(r"C:\Users\v-vvajpai\Desktop\Project_2\datasets\consumer-price-index-cleaned.csv", index=False)